# Cryptographie-04 : Le post-quantique — quand Shor casse RSA

**Série** : [Cryptography](README.md) — le pilier sécurité du hub « Information & calcul » (C# / .NET BCL).

Quatrième carnet de la série, après les briques (01), le passage du canal bruité à l'adversarial (02) et la crypto avancée — homomorphe & ZK (03).

La cryptographie asymétrique qui nous protège aujourd'hui — **RSA** (la factorisation) et **ECC** (le logarithme discret) — repose sur la difficulté, *pour un ordinateur classique*, de deux problèmes algébriques. L'**algorithme de Shor** (1994) les résout en **temps polynomial** sur un ordinateur quantique : la sécurité de ces deux familles s'effondre d'un coup. Le **post-quantique** désigne les schémas dont la dureté ne repose *pas* sur cette structure périodique que Shor exploite.

Ce carnet fait le tour, **calculé** : (1) comment Shor casse RSA ; (2) la famille **lattice** — le problème *Learning-With-Errors*, socle de ML-KEM ; (3) la famille **code-based** (McEliece, renvoi vers InformationTheory-03) ; (4) les **standards NIST** et le prix à payer en taille de clé.

## 1. La menace : ce que Shor casse

RSA et ECC ne sont pas « chiffrés trop fort » : ils reposent sur deux problèmes algébriques — **factoriser un entier** (RSA) et **calculer un logarithme discret** (ECC / Diffie-Hellman) — que l'on croit durs *parce que* leur meilleure attaque connue est exponentielle. Shor change la donne : en exploitant la **périodicité** de ces problèmes via la transformée de Fourier quantique, il les résout en **temps polynomial**. Ce n'est pas une question de puissance de calcul, c'est un **changement de complexité** : le problème que l'on rangeait dans « exponentiel » tombe dans « polynomial ».

Conséquence directe : qui factorise `n` en RSA retrouve la clé privée. C'est le mécanisme, calculé ci-dessous sur une paire RSA miniature.

In [1]:
using System;
using System.Numerics;

// Paire RSA miniature (l'exemple canonique) : n = 3233 = 61 * 53.
// Trop petite pour etre sure en pratique, mais elle suffit a montrer le MECANISME
// de la casse : factoriser n -> phi(n) -> l'exposant prive d.
BigInteger rsaP = 61, rsaQ = 53;
BigInteger rsaN = rsaP * rsaQ;
BigInteger rsaE = 17;
Console.WriteLine($"public : n = {rsaN}, e = {rsaE}");

// La "casse" : factoriser n (brute force, faisable ICI car n est minuscule).
// En production n a 4096 bits : la meme boucle met plus de temps que l'age de
// l'univers... sauf sur un ordinateur quantique, ou Shor la termine en polynomial.
BigInteger rsaPRec = 0, rsaQRec = 0;
for (BigInteger a = 2; a * a <= rsaN; a++)
    if (rsaN % a == 0) { rsaPRec = a; rsaQRec = rsaN / a; break; }
Console.WriteLine($"factorise : n = {rsaPRec} * {rsaQRec}");

// De la factorisation, la cle privee tombe directement :
// phi(n) = (p-1)(q-1), puis d = e^{-1} mod phi(n) (Euclide etendu, inliné).
BigInteger rsaPhi = (rsaPRec - 1) * (rsaQRec - 1);
BigInteger euA = rsaE, euM = rsaPhi, euOldR = euA, euR = euM, euOldS = 1, euS = 0;
while (euR != 0)
{
    BigInteger euT = euOldR / euR;
    (euOldR, euR) = (euR, euOldR - euT * euR);
    (euOldS, euS) = (euS, euOldS - euT * euS);
}
BigInteger rsaD = ((euS % euM) + euM) % euM;
Console.WriteLine($"phi(n) = {rsaPhi}  ->  cle privee d = {rsaD}   [CASSEE]");

The below script needs to be able to find the current output cell; this is an easy method to get it.

public : n = 3233, e = 17


factorise : n = 53 * 61


phi(n) = 3120  ->  cle privee d = 0   [CASSEE]


**Lecture du résultat.** La clé privée `d` n'était « cachée » que par le secret de la factorisation de `n`. Dès que `n` est factorisé, `d` se **recalcule** en une ligne. Toute la sécurité de RSA tenait au fait que factoriser `n` était exponentiellement dur *pour un classique*. Shor rend cette factorisation polynomial : la ligne `d = ModInverse(e, phi)` devient accessible à l'adversaire. C'est exactement le même mécanisme qui tombe pour ECC — le logarithme discret est tout aussi périodique, et tout aussi exploitable par Shor.

In [2]:
using System;
using System.Numerics;

// ============================================================
// Exercice 1 — reconstruire la cle privee
// ============================================================
// On vous donne une AUTRE paire RSA miniature et sa factorisation :
//   ex_n = 77 = 7 * 11,   ex_e = 7
// Completez le calcul de la cle privee :
//   1) exPhi = (p-1)*(q-1)
//   2) exD   = exE^{-1}  mod exPhi
// Indice : la cle privee d verifie  (d * exE) % exPhi == 1.

BigInteger exP = 7, exQ = 11, exE = 7;
BigInteger exN = exP * exQ;
Console.WriteLine($"n = {exN}, e = {exE}");

BigInteger exPhi = 0;   // TODO: (exP - 1) * (exQ - 1)
BigInteger exD   = 0;   // TODO: l'inverse d'exE modulo exPhi

Console.WriteLine($"[exo 1] phi = {exPhi}, d = {exD}");
if (exPhi != 0)
    Console.WriteLine(((exD * exE) % exPhi) == 1
        ? "  -> correct, d est bien l'inverse de e mod phi"
        : "  -> a verifier");
else
    Console.WriteLine("  -> completez exPhi d'abord (phi = 0)");

n = 77, e = 7


[exo 1] phi = 0, d = 0


  -> completez exPhi d'abord (phi = 0)


## 2. La famille lattice : le problème Learning-With-Errors

Le rempart post-quantique le plus standardisé (ML-KEM, ex-Kyber) repose sur un principe **opposé** à RSA : **ajouter du bruit**. Le problème **LWE** (*Learning With Errors*) : donnée une matrice `A` et un vecteur `b = A·s + e` sur un anneau modulo `q`, retrouver le secret `s`, où `e` est un petit vecteur de **bruit** aléatoire.

Pourquoi c'est dur ? Sans le bruit, `b = A·s` est un simple système linéaire : `A` étant (quasi) inversible modulo `q`, on **récupère `s`** par l'algèbre linéaire — c'est exactement le type de structure périodique que Shor adore. Le bruit `e` **masque** cette structure : résoudre naïvement donne `s + A⁻¹·e`, plus `s`. La dureté post-quantique de LWE se **réduit** au problème du *plus court vecteur d'un treillis*, connu dur même pour un quantique. C'est l'anti-structural, calculé ci-dessous en version 2D lisible.

In [3]:
using System;

// LWE en 2D lisible, modulo lq. On montre d'abord le cas SANS bruit :
// b = A*s est un systeme lineaire -> s est recupere par l'inverse de A.
int lq = 17;
int s0 = 5, s1 = 3;                       // le secret
int a00 = 3, a01 = 7, a10 = 5, a11 = 11;  // la matrice A
int det = ((a00 * a11 - a01 * a10) % lq + lq) % lq;
int detInv = 0;
for (int x = 1; x < lq; x++)
    if ((det * x) % lq == 1) { detInv = x; break; }

int b0 = (a00 * s0 + a01 * s1) % lq;
int b1 = (a10 * s0 + a11 * s1) % lq;
Console.WriteLine($"A = [[{a00},{a01}],[{a10},{a11}]],  det(A) = {det} (inversible mod {lq})");

// resolution naive (inverse de A applique a b) : on recupere s exact
int r0 = ((detInv * ( a11 * b0 - a01 * b1)) % lq + lq) % lq;
int r1 = ((detInv * (-a10 * b0 + a00 * b1)) % lq + lq) % lq;
Console.WriteLine($"SANS bruit : b = [{b0},{b1}]  ->  s recupere = ({r0},{r1})  [== secret]");

A = [[3,7],[5,11]],  det(A) = 15 (inversible mod 17)


SANS bruit : b = [2,7]  ->  s recupere = (5,3)  [== secret]


In [4]:
using System;

// AVEC bruit : b' = A*s + e. Le bruit masque la structure : la resolution naive
// ne donne plus s, mais s + A^{-1}*e.
int nse0 = 1, nse1 = -1;                        // petit bruit
int nbp0 = ((b0 + nse0) % lq + lq) % lq;
int nbp1 = ((b1 + nse1) % lq + lq) % lq;
int np0 = ((detInv * ( a11 * nbp0 - a01 * nbp1)) % lq + lq) % lq;
int np1 = ((detInv * (-a10 * nbp0 + a00 * nbp1)) % lq + lq) % lq;
Console.WriteLine($"AVEC bruit : b' = [{nbp0},{nbp1}]  ->  s recupere = ({np0},{np1})  [!= secret ({s0},{s1})]");
Console.WriteLine("  le bruit a MASQUE s : c'est toute la durete de LWE (reduite au plus court vecteur de treillis).");

AVEC bruit : b' = [3,6]  ->  s recupere = (13,7)  [!= secret (5,3)]


  le bruit a MASQUE s : c'est toute la durete de LWE (reduite au plus court vecteur de treillis).


**Lecture du résultat.** Sans bruit, la résolution naïve restitue `s` **exactement** — l'adversaire (classique *ou* quantique) n'a qu'à inverser `A`. Avec le bruit `e`, la même résolution donne un point de chute **faux** au lieu du secret : le bruit, passé à travers `A⁻¹`, **déplace** le résultat. C'est le cœur de LWE — et de tous les schémas lattice : on rend le problème **non-structuré**. La dureté ne repose plus sur une périodicité exploitable par Shor, mais sur le *plus court vecteur d'un treillis*, pour lequel aucune attaque quantique polynomialle n'est connue.

In [5]:
using System;

// ============================================================
// Exercice 2 — le bruit change tout
// ============================================================
// Rejouez le calcul avec VOTRE propre bruit. Choisissez xe0 et xe1 (petits, non
// nuls) et observez que le s "recupere" s'eloigne du secret.
// Indice : le s "recupere" vaut s + A^{-1}*e ; plus e est grand, plus l'erreur croit.

int xe0 = 0, xe1 = 0;   // TODO: mettez par exemple xe0 = 2, xe1 = -1
int xbp0 = ((b0 + xe0) % lq + lq) % lq;
int xbp1 = ((b1 + xe1) % lq + lq) % lq;
int xp0 = ((detInv * ( a11 * xbp0 - a01 * xbp1)) % lq + lq) % lq;
int xp1 = ((detInv * (-a10 * xbp0 + a00 * xbp1)) % lq + lq) % lq;
Console.WriteLine($"[exo 2] e = [{xe0},{xe1}]  ->  s recupere = ({xp0},{xp1})   secret = ({s0},{s1})");
Console.WriteLine((xp0 == s0 && xp1 == s1) ? "  (e nul : s toujours visible)" : "  (e non nul : s masque)");

[exo 2] e = [0,0]  ->  s recupere = (5,3)   secret = (5,3)


  (e nul : s toujours visible)


## 3. La famille code-based : McEliece

Deuxième famille post-quantique majeure, et le **pont** avec la série sœur : la crypto **à base de codes** (McEliece, 1978) s'appuie sur les **codes correcteurs** de la série InformationTheory (sœur, hub Information et calcul) — le même objet que le codage de canal.

L'asymétrie qui fait la sécurité : **encoder** est trivial (multiplier le message par la matrice génératrice `G`, opération linéaire), mais **décoder** un mot reçu bruité *sans connaître la structure cachée* de `G` (la permutation qui le brouille) est le **problème du décodage générique**, NP-difficile. On publie `G` brouillé, on garde la permutation en secret. La théorie des codes vit en InformationTheory ; ici on calcule l'asymétrie encode / décode.

In [6]:
using System;

// Un code lineaire (n=7, k=4) : encoder = G * message (trivial, linéaire, sur GF(2)).
int cn = 7, k = 4;
int[,] G = {
  {1,0,0,0,1,1,1},
  {0,1,0,0,1,1,0},
  {0,0,1,0,1,0,1},
  {0,0,0,1,0,1,1}
};
int[] msg = {1,0,1,1};
int[] cw = new int[cn];
for (int j = 0; j < cn; j++)
{
  int acc = 0;
  for (int i = 0; i < k; i++) acc = (acc + G[i,j] * msg[i]) % 2;
  cw[j] = acc;
}
Console.WriteLine($"encoder (TRIVIAL) : message [{string.Join("", msg)}]  ->  code-mot [{string.Join("", cw)}]");
Console.WriteLine("decoder un mot BRUITE sans la structure cachee de G = decodage generique = NP-difficile.");

encoder (TRIVIAL) : message [1011]  ->  code-mot [1011001]


decoder un mot BRUITE sans la structure cachee de G = decodage generique = NP-difficile.


In [7]:
using System;

// ============================================================
// Exercice 3 — cacher la structure
// ============================================================
// L'astuce de McEliece : publier G brouille par une PERMUTATION des colonnes,
// pas G lui-meme. Appliquez la permutation perm aux colonnes de G, puis
// re-encodez msg : le code-mot change d'aspect, mais l'envoyeur (qui connait la
// permutation) sait le decoder.
// Indice : Gb[i, perm[j]] = G[i, j]  (la colonne j de G part en position perm[j]).

int[] perm = {3, 1, 4, 0, 5, 2, 6};   // une permutation de {0..6}
int[,] Gb = new int[k, cn];
for (int i = 0; i < k; i++)
    for (int j = 0; j < cn; j++)
        Gb[i, j] = 0;   // TODO: Gb[i, perm[j]] = G[i, j]
int[] cw2 = new int[cn];
for (int j = 0; j < cn; j++)
{
  int acc = 0;
  for (int i = 0; i < k; i++) acc = (acc + Gb[i,j] * msg[i]) % 2;
  cw2[j] = acc;
}
Console.WriteLine($"[exo 3] G brouille -> code-mot [{string.Join("", cw2)}]   (aspect change, structure cachee)");

[exo 3] G brouille -> code-mot [0000000]   (aspect change, structure cachee)


## 4. Les standards NIST — et le prix à payer : la taille

Le NIST a **finalisé en 2024** trois standards post-quantiques :

| Standard | Famille | Usage |
|---|---|---|
| **ML-KEM** (ex-Kyber) | lattice | chiffrement / échange de clés |
| **ML-DSA** (ex-Dilithium) | lattice | signature |
| **SLH-DSA** (SPHINCS+) | hash-based | signature |

Le post-quantique n'est pas « meilleur », il est **différemment dur**. Le prix concret : les clés publiques et les signatures PQC sont **nettement plus grosses** qu'en ECC. C'est le compromis à internaliser — la migration (TLS, X.509, les cartes à puce) se cale sur ces tailles, calculées ci-dessous.

In [8]:
using System;

// Le prix du post-quantique, en octets : les cles / signatures PQC sont plus
// grosses qu'en ECC. Chiffres des standards NIST finalises (2024).
var schemes = new (string nom, int clePub, int signature, string note)[]
{
  ("ECC P-256",    64,    64,  "elliptique  (CASSE par Shor)"),
  ("RSA-2048",    256,   256,  "factorisation (CASSE par Shor)"),
  ("ML-KEM-768", 1184,     0,  "lattice  (post-quantique)"),
  ("ML-DSA-65",  1952,  2405,  "lattice  (post-quantique)"),
};
Console.WriteLine($"{"scheme",-14}{"cle pub (o)",-12}{"signature (o)",-16}   famille");
foreach (var s in schemes)
    Console.WriteLine($"{s.nom,-14}{s.clePub,-12}{(s.signature == 0 ? "-" : s.signature.ToString()),-16}   {s.note}");

double ratio = 1184.0 / 64.0;
Console.WriteLine($"\nune cle ML-KEM-768 pèse ~{ratio:0.0}x une cle ECC P-256  (le prix du post-quantique).");

scheme        cle pub (o) signature (o)      famille


ECC P-256     64          64                 elliptique  (CASSE par Shor)


RSA-2048      256         256                factorisation (CASSE par Shor)


ML-KEM-768    1184        -                  lattice  (post-quantique)


ML-DSA-65     1952        2405               lattice  (post-quantique)



une cle ML-KEM-768 pèse ~18,5x une cle ECC P-256  (le prix du post-quantique).


In [9]:
using System;

// ============================================================
// Exercice 4 — le ratio a calculer
// ============================================================
// Combien de fois une SIGNATURE ML-DSA-65 est-elle plus grosse qu'une signature
// ECC P-256 ? Completez le calcul a partir du tableau ci-dessus.
// Indice : ratio = signature_PQC / signature_ECC.

int sigPQC = 0;   // TODO: taille de la signature ML-DSA-65 (cf. tableau)
int sigECC = 64;  // taille de la signature ECC P-256
double xratio = sigECC == 0 ? 0 : sigPQC / (double) sigECC;
Console.WriteLine($"[exo 4] signature ML-DSA-65 / ECC P-256 = {xratio:0.0}x");

[exo 4] signature ML-DSA-65 / ECC P-256 = 0,0x


## En guise de conclusion

Le post-quantique n'est pas une « meilleure » cryptographie : c'est une cryptographie dont la **dureté change de nature**. RSA et ECC reposaient sur une **structure périodique** (factorisation, log discret) que Shor exploite ; les schémas PQC reposent sur des problèmes **non-structurés** (plus court vecteur de treillis, décodage générique) que rien, à ce jour, ne résout en polynomial — quantique compris.

Le prix est lisible dans le tableau : des clés et signatures plus grosses. La migration est un **choix d'ingénierie** (taille vs. durée de vie des secrets) plus qu'une urgence — les ordinateurs quantiques capables d'exécuter Shor à l'échelle nécessaire (des milliers de qubits *stabilisés*) ne sont pas là. Mais les secrets chiffrés **aujourd'hui** et lus **demain** (le *harvest-now, decrypt-later*) justifient de commencer la transition maintenant. C'est le sens de la standardisation NIST 2024.

**Renvois** : InformationTheory (sœur, hub Information et calcul) pour la théorie des codes qui sous-tend McEliece ; le pilier commun [SmartContracts / Privacy](../../SymbolicAI/SmartContracts/) pour la crypto sur chaîne.